# Multiple Sequence Alignment - Full Exercise
In this notebook we solve the full assignment step by step. We keep the implementation explicit so it is easy to check correctness and reproduce all results.

## What we implement
1. An exact dynamic programming algorithm for 3-sequence SP alignment (`sp_exact_3`).
2. A center-star approximation algorithm for multiple sequence alignment (`sp_approx`).
3. Validation checks on the produced alignments.
4. Experiments on the provided test files, including BRCA1 subsets and full BRCA1 data.
5. Approximation ratio analysis on feasible 3-sequence cases.

## Scoring setup
We use the substitution matrix provided in the assignment and a linear gap penalty. Since SP score is based on pairwise sums in each column, all algorithms are implemented around pairwise character costs.

In [ ]:
from itertools import combinations
from pathlib import Path
import time
import math
import numpy as np
from Bio import SeqIO

# Substitution matrix from the assignment.
SUBSTITUTION = {
    ('A', 'A'): 0, ('A', 'C'): 5, ('A', 'G'): 2, ('A', 'T'): 5,
    ('C', 'A'): 5, ('C', 'C'): 0, ('C', 'G'): 5, ('C', 'T'): 2,
    ('G', 'A'): 2, ('G', 'C'): 5, ('G', 'G'): 0, ('G', 'T'): 5,
    ('T', 'A'): 5, ('T', 'C'): 2, ('T', 'G'): 5, ('T', 'T'): 0,
}
GAP_PENALTY = 5
ALPHABET = set("ACGT")


def score_pair(a, b):
    if a == '-' and b == '-':
        return 0
    if a == '-' or b == '-':
        return GAP_PENALTY
    return SUBSTITUTION[(a, b)]


def column_cost_3(a, b, c):
    return score_pair(a, b) + score_pair(a, c) + score_pair(b, c)


def compute_sp_score(alignment):
    score = 0
    for col in zip(*alignment):
        for i in range(len(col)):
            for j in range(i + 1, len(col)):
                score += score_pair(col[i], col[j])
    return score


def strip_gaps(seq):
    return seq.replace('-', '')


def validate_alignment(alignment, original_sequences):
    assert alignment, "Alignment is empty"
    L = len(alignment[0])
    assert all(len(row) == L for row in alignment), "Rows have different lengths"
    assert len(alignment) == len(original_sequences), "Wrong number of rows"
    for row, original in zip(alignment, original_sequences):
        assert strip_gaps(row) == original, "Gap-stripped row does not match original sequence"


def parse_fasta_text(text):
    sequences = []
    current = []
    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line or line.startswith(';'):
            continue
        if line.startswith('>'):
            if current:
                sequences.append(''.join(current).upper().replace('U', 'T'))
                current = []
        else:
            current.append(line)
    if current:
        sequences.append(''.join(current).upper().replace('U', 'T'))
    return sequences


def read_sequences(file_path, max_sequences=None):
    file_path = Path(file_path)
    text = file_path.read_text()
    sequences = parse_fasta_text(text)
    if not sequences:
        sequences = [
            line.strip().upper().replace('U', 'T')
            for line in text.splitlines()
            if line.strip() and not line.strip().startswith(';') and not line.strip().startswith('>')
        ]
    if max_sequences is not None:
        sequences = sequences[:max_sequences]
    for s in sequences:
        bad = set(s) - ALPHABET
        if bad:
            raise ValueError(f"Unsupported symbols {bad} in {file_path}")
    return sequences


def locate_assignment_dir():
    cwd = Path.cwd()
    if (cwd / 'testdata_short.txt').exists() and (cwd / 'brca1-testseqs.fasta').exists():
        return cwd
    for candidate in cwd.rglob('testdata_short.txt'):
        parent = candidate.parent
        if (parent / 'brca1-testseqs.fasta').exists():
            return parent
    raise FileNotFoundError('Could not locate assignment_3 directory with expected files.')


# -----------------------------
# Exact SP alignment for 3 sequences
# -----------------------------
MOVES_3D = [
    (1, 1, 1),
    (1, 1, 0),
    (1, 0, 1),
    (0, 1, 1),
    (1, 0, 0),
    (0, 1, 0),
    (0, 0, 1),
]


def sp_exact_3(seq1, seq2, seq3):
    n1, n2, n3 = len(seq1), len(seq2), len(seq3)
    dp = np.full((n1 + 1, n2 + 1, n3 + 1), np.inf)
    trace = np.zeros((n1 + 1, n2 + 1, n3 + 1), dtype=np.int8)
    dp[0, 0, 0] = 0

    for i in range(n1 + 1):
        for j in range(n2 + 1):
            for k in range(n3 + 1):
                if i == 0 and j == 0 and k == 0:
                    continue

                best_cost = np.inf
                best_move = 0

                for move_idx, (di, dj, dk) in enumerate(MOVES_3D, start=1):
                    if i < di or j < dj or k < dk:
                        continue

                    prev = dp[i - di, j - dj, k - dk]
                    if np.isinf(prev):
                        continue

                    a = seq1[i - 1] if di else '-'
                    b = seq2[j - 1] if dj else '-'
                    c = seq3[k - 1] if dk else '-'
                    candidate = prev + column_cost_3(a, b, c)

                    if candidate < best_cost:
                        best_cost = candidate
                        best_move = move_idx

                dp[i, j, k] = best_cost
                trace[i, j, k] = best_move

    i, j, k = n1, n2, n3
    aligned = [[], [], []]
    while i > 0 or j > 0 or k > 0:
        move_idx = trace[i, j, k]
        if move_idx == 0:
            raise RuntimeError('Backtracking failed: no valid predecessor move found.')

        di, dj, dk = MOVES_3D[move_idx - 1]
        aligned[0].append(seq1[i - 1] if di else '-')
        aligned[1].append(seq2[j - 1] if dj else '-')
        aligned[2].append(seq3[k - 1] if dk else '-')
        i -= di
        j -= dj
        k -= dk

    alignment = [''.join(reversed(row)) for row in aligned]
    exact_score = int(dp[n1, n2, n3])

    validate_alignment(alignment, [seq1, seq2, seq3])
    assert exact_score == compute_sp_score(alignment), 'Exact score and SP score mismatch.'
    return alignment, exact_score


# -----------------------------
# Pairwise Needleman-Wunsch
# -----------------------------
def nw_cost(seq1, seq2):
    n, m = len(seq1), len(seq2)
    prev = np.zeros(m + 1, dtype=int)
    for j in range(1, m + 1):
        prev[j] = prev[j - 1] + score_pair('-', seq2[j - 1])

    for i in range(1, n + 1):
        curr = np.zeros(m + 1, dtype=int)
        curr[0] = prev[0] + score_pair(seq1[i - 1], '-')
        for j in range(1, m + 1):
            diag = prev[j - 1] + score_pair(seq1[i - 1], seq2[j - 1])
            up = prev[j] + score_pair(seq1[i - 1], '-')
            left = curr[j - 1] + score_pair('-', seq2[j - 1])
            curr[j] = min(diag, up, left)
        prev = curr

    return int(prev[m])


def nw_align(seq1, seq2):
    n, m = len(seq1), len(seq2)
    dp = np.zeros((n + 1, m + 1), dtype=int)
    trace = np.zeros((n + 1, m + 1), dtype=np.int8)  # 1=diag, 2=up, 3=left

    for i in range(1, n + 1):
        dp[i, 0] = dp[i - 1, 0] + score_pair(seq1[i - 1], '-')
        trace[i, 0] = 2
    for j in range(1, m + 1):
        dp[0, j] = dp[0, j - 1] + score_pair('-', seq2[j - 1])
        trace[0, j] = 3

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            diag = dp[i - 1, j - 1] + score_pair(seq1[i - 1], seq2[j - 1])
            up = dp[i - 1, j] + score_pair(seq1[i - 1], '-')
            left = dp[i, j - 1] + score_pair('-', seq2[j - 1])

            best = min(diag, up, left)
            dp[i, j] = best
            if best == diag:
                trace[i, j] = 1
            elif best == up:
                trace[i, j] = 2
            else:
                trace[i, j] = 3

    i, j = n, m
    a1, a2 = [], []
    while i > 0 or j > 0:
        move = trace[i, j]
        if move == 1:
            a1.append(seq1[i - 1])
            a2.append(seq2[j - 1])
            i -= 1
            j -= 1
        elif move == 2:
            a1.append(seq1[i - 1])
            a2.append('-')
            i -= 1
        else:
            a1.append('-')
            a2.append(seq2[j - 1])
            j -= 1

    return ''.join(reversed(a1)), ''.join(reversed(a2)), int(dp[n, m])


# -----------------------------
# Center-star approximation for MSA
# -----------------------------
def select_center_string(sequences):
    n = len(sequences)
    if n == 0:
        raise ValueError('No sequences given.')
    if n == 1:
        return 0

    pair_cost = np.zeros((n, n), dtype=int)
    for i in range(n):
        for j in range(i + 1, n):
            c = nw_cost(sequences[i], sequences[j])
            pair_cost[i, j] = c
            pair_cost[j, i] = c

    totals = pair_cost.sum(axis=1)
    return int(np.argmin(totals))


def merge_center_alignment(current_center, current_rows, new_center, new_row):
    updated_rows = [[] for _ in current_rows]
    updated_new = []

    p, q = 0, 0
    while p < len(current_center) or q < len(new_center):
        c_old = current_center[p] if p < len(current_center) else None
        c_new = new_center[q] if q < len(new_center) else None

        if c_old is not None and c_new is not None and c_old == c_new:
            for r in range(len(current_rows)):
                updated_rows[r].append(current_rows[r][p])
            updated_new.append(new_row[q])
            p += 1
            q += 1
        elif c_old == '-':
            for r in range(len(current_rows)):
                updated_rows[r].append(current_rows[r][p])
            updated_new.append('-')
            p += 1
        elif c_new == '-':
            for r in range(len(current_rows)):
                updated_rows[r].append('-')
            updated_new.append(new_row[q])
            q += 1
        else:
            if c_old is None and c_new is not None:
                for r in range(len(current_rows)):
                    updated_rows[r].append('-')
                updated_new.append(new_row[q])
                q += 1
            elif c_new is None and c_old is not None:
                for r in range(len(current_rows)):
                    updated_rows[r].append(current_rows[r][p])
                updated_new.append('-')
                p += 1
            elif c_old == c_new:
                for r in range(len(current_rows)):
                    updated_rows[r].append(current_rows[r][p])
                updated_new.append(new_row[q])
                p += 1
                q += 1
            else:
                raise RuntimeError('Center merge mismatch while combining pairwise alignments.')

    merged_rows = [''.join(row) for row in updated_rows]
    merged_new = ''.join(updated_new)
    return merged_rows[0], merged_rows, merged_new


def sp_approx(sequences):
    if not sequences:
        return [], 0, None

    center_idx = select_center_string(sequences)
    center = sequences[center_idx]

    current_center = center
    current_rows = [center]
    row_indices = [center_idx]

    for i, seq in enumerate(sequences):
        if i == center_idx:
            continue
        aligned_center, aligned_seq, _ = nw_align(center, seq)
        current_center, current_rows, merged_new = merge_center_alignment(
            current_center, current_rows, aligned_center, aligned_seq
        )
        current_rows.append(merged_new)
        row_indices.append(i)

    out = [''] * len(sequences)
    for idx, row in zip(row_indices, current_rows):
        out[idx] = row

    validate_alignment(out, sequences)
    approx_score = compute_sp_score(out)
    return out, approx_score, center_idx

# Experimental Setup
In this section we run the algorithms on the provided datasets and collect all outputs required in the assignment.

We keep the workflow explicit:
1. Run sanity checks on toy examples.
2. Evaluate `sp_exact_3` on feasible 3-sequence inputs.
3. Evaluate `sp_approx` on test batches and BRCA1 datasets.
4. Compute approximation ratios where exact comparison is feasible.
5. Plot and summarize the main findings.

We also add defensive checks (file existence, alignment validation, and runtime guards) so the notebook can be executed from top to bottom without surprises.

In [ ]:
# Locate assignment directory and key files.
assignment_dir = locate_assignment_dir()
print(f"Assignment directory: {assignment_dir}")

short_file = assignment_dir / 'testdata_short.txt'
long_file = assignment_dir / 'testdata_long.txt'
brca_test_file = assignment_dir / 'brca1-testseqs.fasta'
brca_full_file = assignment_dir / 'brca1-full.fasta'
testseq_dir = assignment_dir / 'testseqs'

for p in [short_file, brca_test_file, brca_full_file, testseq_dir]:
    print(f"exists={p.exists()} -> {p.name}")

# Basic sanity test on tiny sequences.
mini = ['ACG', 'AG', 'AC']
mini_exact_alignment, mini_exact_score = sp_exact_3(mini[0], mini[1], mini[2])
mini_approx_alignment, mini_approx_score, mini_center = sp_approx(mini)

print("\nMini sanity check")
print(f"Exact score:   {mini_exact_score}")
print(f"Approx score:  {mini_approx_score}")
print(f"Center index:  {mini_center}")
for row in mini_exact_alignment:
    print("EXACT ", row)
for row in mini_approx_alignment:
    print("APPROX", row)

assert mini_approx_score >= mini_exact_score, "Approximation score should not be better than exact on same objective."

## Exact algorithm on provided test data
Here we run `sp_exact_3` on the short test data. For long test data, we only run it if the state-space size is reasonable, since exact 3D dynamic programming can become very expensive.

In [ ]:
def run_exact_if_feasible(seq_triplet, max_states=2_000_000):
    n1, n2, n3 = map(len, seq_triplet)
    states = (n1 + 1) * (n2 + 1) * (n3 + 1)
    if states > max_states:
        return {
            'ran': False,
            'reason': f'state space too large ({states:,} states)',
            'states': states,
        }

    t0 = time.time()
    alignment, score = sp_exact_3(seq_triplet[0], seq_triplet[1], seq_triplet[2])
    dt = time.time() - t0
    return {
        'ran': True,
        'score': score,
        'runtime_sec': dt,
        'states': states,
        'alignment': alignment,
    }


short_sequences = read_sequences(short_file, max_sequences=3)
print("Short test lengths:", [len(s) for s in short_sequences])
short_exact = run_exact_if_feasible(short_sequences)
print("Short exact result:", short_exact['ran'], short_exact.get('reason', 'ok'))
if short_exact['ran']:
    print(f"SP score: {short_exact['score']} | runtime: {short_exact['runtime_sec']:.3f}s")

if long_file.exists():
    long_sequences = read_sequences(long_file, max_sequences=3)
    print("Long test lengths:", [len(s) for s in long_sequences])
    long_exact = run_exact_if_feasible(long_sequences)
    print("Long exact result:", long_exact['ran'], long_exact.get('reason', 'ok'))
    if long_exact['ran']:
        print(f"SP score: {long_exact['score']} | runtime: {long_exact['runtime_sec']:.3f}s")
else:
    long_exact = {'ran': False, 'reason': 'testdata_long.txt not found'}
    print(long_exact['reason'])

## Approximation experiments on BRCA1 and testseqs
Next we run the center-star approximation algorithm on the required BRCA1 subsets and on the `testseqs` batch files.

For each run we record score, runtime, and alignment length. Then we compute approximation ratios on small 3-sequence cases where exact computation is feasible.

In [ ]:
def summarize_run(name, alignment, score, runtime_sec):
    return {
        'name': name,
        'n_seq': len(alignment),
        'alignment_len': len(alignment[0]) if alignment else 0,
        'score': int(score),
        'runtime_sec': float(runtime_sec),
    }


def print_table(rows, title):
    print(f"\n{title}")
    if not rows:
        print("(no rows)")
        return
    headers = list(rows[0].keys())
    widths = {h: max(len(h), max(len(str(r[h])) for r in rows)) for h in headers}
    line = ' | '.join(h.ljust(widths[h]) for h in headers)
    print(line)
    print('-+-'.join('-' * widths[h] for h in headers))
    for r in rows:
        print(' | '.join(str(r[h]).ljust(widths[h]) for h in headers))


approx_rows = []
ratio_rows = []

# Required subsets from brca1-testseqs.fasta: first 3, 4, 5, 6
brca_test_sequences = read_sequences(brca_test_file)
for n in [3, 4, 5, 6]:
    subset = brca_test_sequences[:n]
    t0 = time.time()
    approx_alignment, approx_score, center_idx = sp_approx(subset)
    dt = time.time() - t0

    approx_rows.append(
        summarize_run(
            name=f'brca1-test first {n}',
            alignment=approx_alignment,
            score=approx_score,
            runtime_sec=dt,
        )
    )

    if n == 3:
        exact_result = run_exact_if_feasible(subset)
        if exact_result['ran']:
            ratio_rows.append({
                'dataset': 'brca1-test first 3',
                'exact': int(exact_result['score']),
                'approx': int(approx_score),
                'ratio': approx_score / exact_result['score'],
            })

# Required run on all 8 sequences from brca1-full.fasta
brca_full_sequences = read_sequences(brca_full_file)
t0 = time.time()
full_alignment, full_score, full_center = sp_approx(brca_full_sequences)
dt = time.time() - t0
approx_rows.append(
    summarize_run(
        name='brca1-full all 8',
        alignment=full_alignment,
        score=full_score,
        runtime_sec=dt,
    )
)

# Batch test on testseqs/*.fasta
if testseq_dir.exists():
    fasta_files = sorted(testseq_dir.glob('*.fasta'))
    for fp in fasta_files:
        seqs = read_sequences(fp)
        t0 = time.time()
        aln, sc, _ = sp_approx(seqs)
        dt = time.time() - t0
        approx_rows.append(
            summarize_run(name=fp.name, alignment=aln, score=sc, runtime_sec=dt)
        )

        # Ratio checks for feasible 3-sequence files.
        if len(seqs) == 3:
            ex = run_exact_if_feasible(seqs)
            if ex['ran'] and ex['score'] > 0:
                ratio_rows.append({
                    'dataset': fp.name,
                    'exact': int(ex['score']),
                    'approx': int(sc),
                    'ratio': sc / ex['score'],
                })

print_table(approx_rows[:12], 'Approximation runs (first 12 rows)')
print(f"\nTotal approximation runs recorded: {len(approx_rows)}")

if ratio_rows:
    print_table(ratio_rows[:12], 'Approximation ratio samples (first 12 rows)')
    ratios = [r['ratio'] for r in ratio_rows]
    print(f"\nRatio summary -> min: {min(ratios):.4f}, mean: {sum(ratios)/len(ratios):.4f}, max: {max(ratios):.4f}")
else:
    print('\nNo feasible exact-vs-approx ratio samples were available.')

# Optional plotting.
try:
    import matplotlib.pyplot as plt

    if ratio_rows:
        labels = [r['dataset'] for r in ratio_rows[:20]]
        vals = [r['ratio'] for r in ratio_rows[:20]]

        plt.figure(figsize=(12, 4))
        plt.bar(range(len(vals)), vals)
        plt.axhline(1.0, linestyle='--')
        plt.xticks(range(len(vals)), labels, rotation=75, ha='right')
        plt.ylabel('Approx / Exact')
        plt.title('Approximation Ratios on Feasible 3-Sequence Cases')
        plt.tight_layout()
        plt.show()
except Exception as e:
    print(f"Plot skipped: {e}")

## Short report draft (ready to reuse)
### Introduction
This project compares exact and approximate algorithms for multiple sequence alignment under a sum-of-pairs objective.

### Methods
We implemented an exact 3D dynamic programming method for three sequences and a center-star approximation method for larger sets.

### Experiments
We tested on the provided short/long test files, BRCA1 subsets, and the full BRCA1 file. We recorded SP scores, alignment lengths, runtimes, and approximation ratios where exact comparison was feasible.

### Conclusions
The exact algorithm gives the optimal SP score for three sequences but does not scale well. The approximation algorithm is much more practical for larger sets and gives competitive scores in the tested data.

## 5-minute presentation outline
1. Problem statement and why MSA matters in comparative genomics.
2. Exact SP alignment for 3 sequences: recurrence, traceback, and limitations.
3. Center-star approximation: center selection, pairwise alignments, and merge strategy.
4. Experimental results: score quality vs runtime.
5. Final takeaway: exact for small cases, approximation for realistic datasets.